# 04 — Fourier Neural Operator (FNO)

**What FNO is**: An amortized operator that learns the mapping from input functions
(initial conditions) to output functions (solutions at time T). Once trained,
each new instance costs one forward pass.

**Architecture spec** (from problem statement):
- 4 Fourier layers, 16 modes, width 64
- Batch 20, Adam 1e-3, ~200 epochs on 1000 instances

**Why FNO is perfect for heat equation**: The true solution operator IS a Fourier-space
multiplication — each mode k decays by exp(-alpha*(2pi*k)^2*t). The FNO layer does
exactly this: learn weights in Fourier space.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import math
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'FNO using device: {device}')

In [ ]:
# ============================================================
# FNO Architecture (built from scratch per project rules)
# ============================================================

class FourierLayer1d(nn.Module):
    """
    Single Fourier layer: the core FNO building block.

    Operation:
        1. FFT the input along the spatial axis
        2. Keep only the lowest `modes` frequencies (truncate high freq)
        3. Multiply those modes by learned complex weights R
        4. Inverse FFT back to physical space
        5. Add a local linear term W*x (like a 1x1 conv)
        6. Apply nonlinearity

    The truncation (step 2) is what makes FNO cheap and resolution-invariant:
    the same weights apply regardless of how many grid points are used.
    """

    def __init__(self, width, modes):
        super().__init__()
        self.width = width
        self.modes = modes
        # Complex Fourier weights: shape (width, width, modes)
        scale = 1.0 / (width * width)
        self.weights_real = nn.Parameter(scale * torch.randn(width, width, modes))
        self.weights_imag = nn.Parameter(scale * torch.randn(width, width, modes))
        # Local linear term
        self.W = nn.Linear(width, width)

    def forward(self, x):
        # x shape: (batch, nx, width)
        B, nx, C = x.shape

        # FFT along spatial axis
        x_ft = torch.fft.rfft(x, dim=1)  # shape: (B, nx//2+1, C)

        # Keep only lowest `modes` frequencies
        out_ft = torch.zeros(B, nx // 2 + 1, C, device=x.device, dtype=torch.cfloat)

        # Complex multiplication: sum over input channels
        R = torch.complex(self.weights_real, self.weights_imag)  # (C_out, C_in, modes)
        # x_ft[:, :modes, :] has shape (B, modes, C)
        # We want (B, modes, C_out) = einsum(bcm, oim -> bio)
        # but let's keep it simple:
        xf_low = x_ft[:, :self.modes, :]  # (B, modes, C_in)
        # (B, modes, C_out) via einsum: b=batch, m=mode, i=in_channel, o=out_channel
        out_ft[:, :self.modes, :] = torch.einsum('bmi,iom->bmo', xf_low, R)

        # Inverse FFT
        x_spectral = torch.fft.irfft(out_ft, n=nx, dim=1)  # (B, nx, C)

        # Local linear + combine
        x_local = self.W(x)  # (B, nx, C)
        return x_spectral + x_local


class FNO1d(nn.Module):
    """
    1D Fourier Neural Operator for operator learning.

    Input:  (batch, nx, 1)  — initial condition u0(x)
    Output: (batch, nx, 1)  — solution at time T: u_T(x)

    Architecture:
        1. Lift: 1 -> width channels
        2. n_layers Fourier layers with GELU activations
        3. Project: width -> 1 channel
    """

    def __init__(self, modes=16, width=64, n_layers=4):
        super().__init__()
        self.modes  = modes
        self.width  = width

        # Lifting layer: maps 1 channel to `width` channels
        self.lift = nn.Linear(1, width)

        # Fourier layers
        self.fourier_layers = nn.ModuleList(
            [FourierLayer1d(width, modes) for _ in range(n_layers)]
        )
        self.norms = nn.ModuleList(
            [nn.LayerNorm(width) for _ in range(n_layers)]
        )

        # Projection back to 1 output channel
        self.proj1 = nn.Linear(width, 128)
        self.proj2 = nn.Linear(128, 1)

    def forward(self, x):
        # x: (B, nx, 1)
        x = self.lift(x)  # (B, nx, width)
        for fl, norm in zip(self.fourier_layers, self.norms):
            x = F.gelu(norm(fl(x)))
        x = F.gelu(self.proj1(x))
        x = self.proj2(x)   # (B, nx, 1)
        return x


# Count parameters
model_demo = FNO1d(modes=16, width=64)
n_params = sum(p.numel() for p in model_demo.parameters())
print(f'FNO architecture ready. Parameter count: {n_params:,}')

In [ ]:
# ============================================================
# Data Generation for Heat Equation Operator
# ============================================================

def generate_heat_ic(k_max=16, seed=42):
    """Generate a single Fourier-series IC with given seed."""
    rng = torch.Generator()
    rng.manual_seed(seed)
    x = torch.linspace(0, 1, 256)
    u = torch.zeros(256)
    for k in range(1, k_max + 1):
        a_k = torch.randn(1, generator=rng).item() * k**-2
        b_k = torch.randn(1, generator=rng).item() * k**-2
        u += a_k * torch.sin(2*math.pi*k*x) + b_k * torch.cos(2*math.pi*k*x)
    return u


def generate_heat_dataset(
    num_samples=1000, nx=256, alpha=0.01, T=1.0,
    alpha_decay=2, seed_train=0
):
    """
    Generate (IC, solution_at_T) pairs for the heat equation.

    Data design from problem statement:
    - 1024-point master grid, subsampled to nx
    - alpha_decay=2 for training (smooth), alpha_decay=1 for OOD test (rough)
    - Fixed seeds for reproducibility

    Args:
        num_samples: number of IC-solution pairs
        nx: number of grid points (subset of 1024)
        alpha: thermal diffusivity
        T: final time
        alpha_decay: controls IC smoothness (a_k ~ k^-alpha_decay)
        seed_train: base seed for the training set

    Returns:
        inputs: (num_samples, nx, 1)
        outputs: (num_samples, nx, 1)
    """
    # Generate on 1024 master grid, then subsample
    nx_master = 1024
    step = nx_master // nx
    x_master = torch.linspace(0, 1, nx_master)

    inputs  = torch.zeros(num_samples, nx)
    outputs = torch.zeros(num_samples, nx)

    for i in range(num_samples):
        rng = torch.Generator()
        rng.manual_seed(seed_train + i)

        k_max = 16
        u0 = torch.zeros(nx_master)
        uT = torch.zeros(nx_master)

        for k in range(1, k_max + 1):
            a_k = torch.randn(1, generator=rng).item() * k**(-alpha_decay)
            b_k = torch.randn(1, generator=rng).item() * k**(-alpha_decay)
            decay = math.exp(-alpha * (2*math.pi*k)**2 * T)
            wave = a_k * torch.sin(2*math.pi*k*x_master) + b_k * torch.cos(2*math.pi*k*x_master)
            u0 += wave
            uT += wave * decay

        inputs[i]  = u0[::step]
        outputs[i] = uT[::step]

    return inputs.unsqueeze(-1), outputs.unsqueeze(-1)


# Generate datasets
print('Generating training data...')
train_inputs, train_outputs = generate_heat_dataset(num_samples=1000, nx=256, seed_train=0)
print(f'Training set: {train_inputs.shape} -> {train_outputs.shape}')

print('Generating test data (same distribution)...')
test_inputs, test_outputs = generate_heat_dataset(num_samples=200, nx=256, seed_train=10000)
print(f'Test set: {test_inputs.shape} -> {test_outputs.shape}')

print('Generating OOD test data (alpha_decay=1, rougher ICs)...')
ood_inputs, ood_outputs = generate_heat_dataset(num_samples=200, nx=256, alpha_decay=1, seed_train=20000)
print(f'OOD set: {ood_inputs.shape} -> {ood_outputs.shape}')

In [ ]:
# ============================================================
# FNO Training
# ============================================================

def train_fno(
    train_inputs, train_outputs,
    epochs=200, batch_size=20, lr=1e-3, modes=16, width=64
):
    """
    Train the FNO on (IC, solution) pairs.

    Returns trained model and training history.
    """
    dataset = data.TensorDataset(
        train_inputs.to(device), train_outputs.to(device)
    )
    loader = data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = FNO1d(modes=modes, width=width).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

    history = []
    print(f'Training FNO: {epochs} epochs, batch={batch_size}, modes={modes}, width={width}')
    t_start = time.time()

    model.train()
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for x_batch, y_batch in loader:
            optimizer.zero_grad()
            pred = model(x_batch)
            loss = F.mse_loss(pred, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        history.append(avg_loss)
        if epoch % 50 == 0 or epoch == 1:
            print(f'  Epoch {epoch:4d}/{epochs} | MSE Loss: {avg_loss:.6f}')

    elapsed = time.time() - t_start
    print(f'Training complete in {elapsed:.1f}s')
    model.eval()
    return model, history, elapsed


fno_model, fno_history, fno_train_time = train_fno(
    train_inputs, train_outputs, epochs=200, batch_size=20
)

# Loss curve
plt.figure(figsize=(8, 4))
plt.semilogy(fno_history)
plt.title('FNO Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.grid(True, which='both', alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Evaluation: FNO vs Exact on in-distribution test set
# ============================================================

def relative_l2_batch(pred, exact):
    """Mean relative L2 error over a batch."""
    # pred, exact: (B, nx, 1)
    pred_flat  = pred.view(pred.shape[0], -1)
    exact_flat = exact.view(exact.shape[0], -1)
    norms_diff = torch.linalg.norm(pred_flat - exact_flat, dim=1)
    norms_exact = torch.linalg.norm(exact_flat, dim=1)
    return (norms_diff / (norms_exact + 1e-8)).mean().item()


with torch.no_grad():
    test_pred = fno_model(test_inputs.to(device)).cpu()
    ood_pred  = fno_model(ood_inputs.to(device)).cpu()

err_in  = relative_l2_batch(test_pred, test_outputs)
err_ood = relative_l2_batch(ood_pred, ood_outputs)

print(f'=== FNO Evaluation ===')
print(f'In-distribution L2 error  : {err_in:.6f}')
print(f'OOD L2 error (alpha_dec=1): {err_ood:.6f}')

# Visualize a few test samples
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
x_grid = torch.linspace(0, 1, 256)
for col, idx in enumerate([0, 50, 150]):
    # In-distribution
    ax = axes[0, col]
    ax.plot(x_grid.numpy(), test_outputs[idx].squeeze().numpy(), label='Exact', linewidth=2)
    ax.plot(x_grid.numpy(), test_pred[idx].squeeze().numpy(), label='FNO', linestyle='--')
    sample_err = (torch.norm(test_pred[idx] - test_outputs[idx]) /
                  (torch.norm(test_outputs[idx]) + 1e-8)).item()
    ax.set_title(f'In-dist #{idx} (L2={sample_err:.4f})')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # OOD
    ax = axes[1, col]
    ax.plot(x_grid.numpy(), ood_outputs[idx].squeeze().numpy(), label='Exact', linewidth=2)
    ax.plot(x_grid.numpy(), ood_pred[idx].squeeze().numpy(), label='FNO OOD', linestyle='--')
    sample_err_ood = (torch.norm(ood_pred[idx] - ood_outputs[idx]) /
                      (torch.norm(ood_outputs[idx]) + 1e-8)).item()
    ax.set_title(f'OOD #{idx} (L2={sample_err_ood:.4f})')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('FNO: In-distribution vs OOD (rougher IC)')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Validation Test: FNO vs Nearest-Neighbour Baseline
# ============================================================
# If FNO error > NN error, it is memorizing rather than learning an operator.

print('=== FNO Generalization Validation ===')

# Use a fractional-k IC that is strictly out of the training distribution
x_grid = torch.linspace(0, 1, 256)
test_k = 4.5  # half-integer: between integer training modes

test_ic_ood  = torch.sin(test_k * math.pi * x_grid).unsqueeze(-1)  # (256,1)
test_sol_ood = test_ic_ood * math.exp(-0.01 * (test_k*math.pi)**2 * 1.0)

# Nearest-neighbour prediction from training set
dists = torch.norm(
    train_inputs.squeeze(-1) - test_ic_ood.squeeze(-1).unsqueeze(0),
    dim=1
)
nn_idx = dists.argmin()
nn_pred = train_outputs[nn_idx]

# FNO prediction
fno_model.eval()
with torch.no_grad():
    fno_pred_ood = fno_model(test_ic_ood.unsqueeze(0).to(device)).squeeze(0).cpu()

nn_err  = (torch.norm(nn_pred.squeeze() - test_sol_ood.squeeze()) /
           (torch.norm(test_sol_ood.squeeze()) + 1e-8)).item()
fno_err = (torch.norm(fno_pred_ood.squeeze() - test_sol_ood.squeeze()) /
           (torch.norm(test_sol_ood.squeeze()) + 1e-8)).item()

print(f'Test IC: k={test_k} (unseen fractional wavenumber)')
print(f'Nearest-Neighbour L2 error: {nn_err:.4f}')
print(f'FNO L2 error              : {fno_err:.4f}')

if fno_err < nn_err:
    print('\nSUCCESS: FNO generalizes beyond memorization.')
else:
    print('\nWARNING: FNO is not outperforming nearest-neighbour — check training.')